In [3]:
%pip install implicit

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import implicit as imp
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

In [2]:
df_movie = pd.read_csv(r"C:\Users\Tobis\Desktop\Implicit Recommendation System\movie\movie.csv")
df_rating = pd.read_csv(r"C:\Users\Tobis\Desktop\Implicit Recommendation System\rating\rating.csv")

In [3]:
print(df_rating.head())

   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40


In [4]:
print(df_movie.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [5]:
print(max(df_movie["movieId"]))
print(len(df_movie["movieId"]))
print(max(df_movie.index))

131262
27278
27277


In [6]:
df_movie["movieId_index"] = df_movie.index
print(df_movie.head())
print(max(df_movie["movieId"]))
print(len(df_movie["movieId"]))
print(max(df_movie.index))

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  movieId_index  
0  Adventure|Animation|Children|Comedy|Fantasy              0  
1                   Adventure|Children|Fantasy              1  
2                               Comedy|Romance              2  
3                         Comedy|Drama|Romance              3  
4                                       Comedy              4  
131262
27278
27277


In [7]:
# New column "implicit" as positive feedback
df_rating["implicit"] = (df_rating["rating"] >= 4).astype(int)
print(df_rating.head(10))

   userId  movieId  rating            timestamp  implicit
0       1        2     3.5  2005-04-02 23:53:47         0
1       1       29     3.5  2005-04-02 23:31:16         0
2       1       32     3.5  2005-04-02 23:33:39         0
3       1       47     3.5  2005-04-02 23:32:07         0
4       1       50     3.5  2005-04-02 23:29:40         0
5       1      112     3.5  2004-09-10 03:09:00         0
6       1      151     4.0  2004-09-10 03:08:54         1
7       1      223     4.0  2005-04-02 23:46:13         1
8       1      253     4.0  2005-04-02 23:35:40         1
9       1      260     4.0  2005-04-02 23:33:46         1


In [8]:
df_joined = df_rating.merge(
    df_movie,
    on="movieId",
    how="left"
)

print(df_joined.head())

   userId  movieId  rating            timestamp  implicit  \
0       1        2     3.5  2005-04-02 23:53:47         0   
1       1       29     3.5  2005-04-02 23:31:16         0   
2       1       32     3.5  2005-04-02 23:33:39         0   
3       1       47     3.5  2005-04-02 23:32:07         0   
4       1       50     3.5  2005-04-02 23:29:40         0   

                                               title  \
0                                     Jumanji (1995)   
1  City of Lost Children, The (Cité des enfants p...   
2          Twelve Monkeys (a.k.a. 12 Monkeys) (1995)   
3                        Seven (a.k.a. Se7en) (1995)   
4                         Usual Suspects, The (1995)   

                                   genres  movieId_index  
0              Adventure|Children|Fantasy              1  
1  Adventure|Drama|Fantasy|Mystery|Sci-Fi             28  
2                 Mystery|Sci-Fi|Thriller             31  
3                        Mystery|Thriller             46  
4

In [9]:
print(df_joined)

          userId  movieId  rating            timestamp  implicit  \
0              1        2     3.5  2005-04-02 23:53:47         0   
1              1       29     3.5  2005-04-02 23:31:16         0   
2              1       32     3.5  2005-04-02 23:33:39         0   
3              1       47     3.5  2005-04-02 23:32:07         0   
4              1       50     3.5  2005-04-02 23:29:40         0   
...          ...      ...     ...                  ...       ...   
20000258  138493    68954     4.5  2009-11-13 15:42:00         1   
20000259  138493    69526     4.5  2009-12-03 18:31:48         1   
20000260  138493    69644     3.0  2009-12-07 18:10:57         0   
20000261  138493    70286     5.0  2009-11-13 15:42:24         1   
20000262  138493    71619     2.5  2009-10-17 20:25:36         0   

                                                      title  \
0                                            Jumanji (1995)   
1         City of Lost Children, The (Cité des enfants p.

In [10]:
df_positive = df_joined[df_joined["implicit"] == 1].copy()

In [11]:
print(df_positive)

          userId  movieId  rating            timestamp  implicit  \
6              1      151     4.0  2004-09-10 03:08:54         1   
7              1      223     4.0  2005-04-02 23:46:13         1   
8              1      253     4.0  2005-04-02 23:35:40         1   
9              1      260     4.0  2005-04-02 23:33:46         1   
10             1      293     4.0  2005-04-02 23:31:43         1   
...          ...      ...     ...                  ...       ...   
20000256  138493    66762     4.5  2009-10-17 18:50:08         1   
20000257  138493    68319     4.5  2009-12-07 18:15:20         1   
20000258  138493    68954     4.5  2009-11-13 15:42:00         1   
20000259  138493    69526     4.5  2009-12-03 18:31:48         1   
20000261  138493    70286     5.0  2009-11-13 15:42:24         1   

                                                      title  \
6                                            Rob Roy (1995)   
7                                             Clerks (199

In [12]:
#user_ids = df_rating["userId"].unique()
#movie_ids = df_rating["movieId"].unique()

#user_map = {u: i for i, u in enumerate(user_ids)}
#movie_map = {m: i for i, m in enumerate(movie_ids)}

#df_positive["user_idx"] = df_positive["userId"].map(user_map)
#df_positive["movie_idx"] = df_positive["movieId"].map(movie_map)

In [13]:
# Get User_Item matrix
user_item = csr_matrix(
    (
        df_positive["implicit"],
        (
            df_positive["userId"],
            df_positive["movieId_index"]
        )
    )
)
print(user_item)

  (1, 149)	1
  (1, 220)	1
  (1, 250)	1
  (1, 257)	1
  (1, 290)	1
  (1, 293)	1
  (1, 315)	1
  (1, 537)	1
  (1, 1017)	1
  (1, 1057)	1
  (1, 1068)	1
  (1, 1075)	1
  (1, 1171)	1
  (1, 1173)	1
  (1, 1175)	1
  (1, 1188)	1
  (1, 1189)	1
  (1, 1193)	1
  (1, 1212)	1
  (1, 1221)	1
  (1, 1230)	1
  (1, 1231)	1
  (1, 1238)	1
  (1, 1250)	1
  (1, 1292)	1
  :	:
  (138493, 11362)	1
  (138493, 11401)	1
  (138493, 11493)	1
  (138493, 11707)	1
  (138493, 11724)	1
  (138493, 11804)	1
  (138493, 11858)	1
  (138493, 11872)	1
  (138493, 11892)	1
  (138493, 11911)	1
  (138493, 11972)	1
  (138493, 12139)	1
  (138493, 12215)	1
  (138493, 12565)	1
  (138493, 12631)	1
  (138493, 12691)	1
  (138493, 12746)	1
  (138493, 12875)	1
  (138493, 12926)	1
  (138493, 13358)	1
  (138493, 13498)	1
  (138493, 13677)	1
  (138493, 13767)	1
  (138493, 13876)	1
  (138493, 14009)	1


In [14]:
#Train ALS on User_Item matrix
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=20
)

# implicit erwartet Item-User-Matrix
model.fit(user_item)

C:\Users\Tobis\Python\anaconda3\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 6 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [15]:
# Recommendations
user_idx = 1

ids, scores = model.recommend(
    userid=user_idx,
    user_items=user_item[user_idx],
    N=10
)
print(ids)
print(scores)

[1172 1263 1067 1184 6764 1913   49 2486 6271 3702]
[0.57840693 0.5754086  0.49884492 0.48755497 0.4764397  0.45259142
 0.41701752 0.4105261  0.40532646 0.40000296]


In [16]:
# Lookup movies
movie_lookup = (
    df_movie[["movieId_index", "title"]]
    .drop_duplicates()
    .set_index("movieId_index")["title"]
    .to_dict()
)

for movie_id in ids:
    print(movie_id, "->", movie_lookup.get(movie_id, "nicht gefunden"))

1172 -> Princess Bride, The (1987)
1263 -> Indiana Jones and the Last Crusade (1989)
1067 -> Reservoir Dogs (1992)
1184 -> Star Wars: Episode VI - Return of the Jedi (1983)
6764 -> Kill Bill: Vol. 1 (2003)
1913 -> Exorcist, The (1973)
49 -> Usual Suspects, The (1995)
2486 -> Matrix, The (1999)
6271 -> Finding Nemo (2003)
3702 -> X-Men (2000)


In [17]:
# This user has the following recommendations
recommendations = pd.DataFrame({
    "movieId_index": ids,
    "score": scores
})

recommendations = recommendations.merge(
    df_movie[["movieId_index", "title", "genres"]],
    on="movieId_index",
    how="left"
)

print(recommendations)

   movieId_index     score                                              title  \
0           1172  0.578407                         Princess Bride, The (1987)   
1           1263  0.575409          Indiana Jones and the Last Crusade (1989)   
2           1067  0.498845                              Reservoir Dogs (1992)   
3           1184  0.487555  Star Wars: Episode VI - Return of the Jedi (1983)   
4           6764  0.476440                           Kill Bill: Vol. 1 (2003)   
5           1913  0.452591                               Exorcist, The (1973)   
6             49  0.417018                         Usual Suspects, The (1995)   
7           2486  0.410526                                 Matrix, The (1999)   
8           6271  0.405326                                Finding Nemo (2003)   
9           3702  0.400003                                       X-Men (2000)   

                                    genres  
0  Action|Adventure|Comedy|Fantasy|Romance  
1                 

In [18]:
# This user allready whatched the following movies aund rated them positive
df_user1 = df_positive[df_positive["userId"] == user_idx][["userId", "title", "genres"]]
print(df_user1.to_string())

     userId                                                                                           title                                               genres
6         1                                                                                  Rob Roy (1995)                             Action|Drama|Romance|War
7         1                                                                                   Clerks (1994)                                               Comedy
8         1                                       Interview with the Vampire: The Vampire Chronicles (1994)                                         Drama|Horror
9         1                                                       Star Wars: Episode IV - A New Hope (1977)                              Action|Adventure|Sci-Fi
10        1                                  Léon: The Professional (a.k.a. The Professional) (Léon) (1994)                          Action|Crime|Drama|Thriller
11        1                       